# CS-781 Phase 1 — Data preparation (notebook A)

**Kaggle settings:** Accelerator **None (CPU)** · Internet **On** · Add-ons → Secrets → `Capstone_Phase_1` switched on.

What this notebook produces in `/kaggle/working` (used as input by the training notebook):

| File | What |
|---|---|
| `bpe_tok.py` | your Assignment-1 BPE tokenizer + save/load/fast-encode helpers |
| `bpe_full.json`, `bpe.json` | tokenizer trained on legal text (full 32k run, and the size you choose) |
| `train.bin`, `val.bin` | the whole training/validation corpus as `uint16` token ids |
| `val_prompts.json`, `val_truth.json` | 300 test-like examples (512-word prompt → next 128 words) |
| `meta.json` | statistics for your report |

Run cells **top to bottom**. Total ≈ 1 hour. The only decision you make is `VOCAB_SIZE` in A7, after reading the table from A6.

## A1. Setup

In [ ]:
!pip install -q ijson
import os, json, time, random, re, hashlib
from collections import Counter
import numpy as np, requests, ijson
from huggingface_hub import login, hf_hub_url
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("Capstone_Phase_1")
login(token=HF_TOKEN)
REPO = "Exploration-Lab/CS-781-Capstone"
AUTH = {"Authorization": f"Bearer {HF_TOKEN}"}
WORK = "/kaggle/working"   # saved as notebook output -> used by the training notebook
TMP = "/tmp/p1"            # scratch space, not saved
os.makedirs(TMP, exist_ok=True)
SEED = 42
print("ijson backend:", ijson.backend)   # should say yajl2_c

## A2. Data mix and filters
Proposed mix (≈405M words ≈ 500M tokens — about what one T4 can see in ~4.5 h). Supreme Court is used in full because the test prompts read like judgments. Record this choice and the filter thresholds in your report.

In [ ]:
FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
}
# Word totals measured in the exploration notebook
TOTAL_WORDS = {"supreme_court": 210_205_151, "acts": 119_638_348,
               "tribunals": 975_917_446, "high_courts": 1_594_919_242}
# Training words to keep per source (documents are sampled at random across the whole file)
TARGET_WORDS = {"supreme_court": 210_000_000, "acts": 25_000_000,
                "tribunals": 60_000_000, "high_courts": 110_000_000}
MIN_WORDS = 200          # drop very short docs (one-paragraph bail/adjournment orders)
MAX_DEVANAGARI = 0.01    # drop docs where >1% of characters are Devanagari (bilingual orders)
MAX_PIPE_RATE = 0.02     # drop table-heavy docs: more than 1 '|' per 50 words
VAL_DOC_PROB = 0.004     # share of kept docs held out for validation

## A3. Stream → filter → save one document per line (≈12–15 min)

In [ ]:
DEVANAGARI = re.compile(r"[\u0900-\u097F]")

def stream_texts(path):
    url = hf_hub_url(REPO, path, repo_type="dataset")
    with requests.get(url, headers=AUTH, stream=True, timeout=120) as r:
        r.raise_for_status()
        r.raw.decode_content = True
        for rec in ijson.items(r.raw, "item"):
            yield rec.get("text", "") if isinstance(rec, dict) else str(rec)

rng = random.Random(SEED)
seen, stats = set(), {}
for name, path in FILES.items():
    p_keep = min(1.0, 1.25 * TARGET_WORDS[name] / TOTAL_WORDS[name])
    s, t0 = Counter(), time.time()
    with open(f"{TMP}/{name}.train.txt", "w", encoding="utf-8") as ftr, \
         open(f"{TMP}/{name}.val.txt", "w", encoding="utf-8") as fva:
        for text in stream_texts(path):
            s["seen"] += 1
            if s["seen"] % 250_000 == 0:
                print(f"  {name}: {s['seen']:,} seen | {s['train_words']/1e6:,.0f}M train words | {time.time()-t0:,.0f}s", flush=True)
            if rng.random() > p_keep:
                continue
            words = text.split()
            if len(words) < MIN_WORDS:
                s["drop_short"] += 1; continue
            if text.count("|") > MAX_PIPE_RATE * len(words):
                s["drop_table"] += 1; continue
            if len(DEVANAGARI.findall(text)) > MAX_DEVANAGARI * len(text):
                s["drop_devanagari"] += 1; continue
            norm = " ".join(words)                      # same whitespace format as the test prompts
            h = hashlib.md5(norm.encode("utf-8")).digest()
            if h in seen:
                s["drop_duplicate"] += 1; continue
            seen.add(h)
            if rng.random() < VAL_DOC_PROB:
                fva.write(norm + "\n"); s["val_docs"] += 1; s["val_words"] += len(words)
            else:
                ftr.write(norm + "\n"); s["train_docs"] += 1; s["train_words"] += len(words)
                if s["train_words"] >= TARGET_WORDS[name]:
                    break
    s["seconds"], s["p_keep"] = round(time.time() - t0), round(p_keep, 4)
    stats[name] = dict(s)
    print(name, json.dumps(stats[name]))

## A4. Tokenizer module
**A4a** is your own Assignment-1 Question-5 code, copied unchanged from `cs781-a1-soubhagya-raha-231025.ipynb` (cells: `get_word_freqs`, `word_to_symbols`, `BPETokenizer`, `detokenize`). **A4b** appends save/load, vocab truncation and a cached encoder.

In [ ]:
%%writefile bpe_tok.py
# ===== Assignment 1, Question 5: BPE tokenizer (Soubhagya Raha) — copied unchanged =====
from collections import Counter, defaultdict
from tqdm.auto import tqdm

SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]


def get_word_freqs(texts):
    word_freqs = Counter()
    for text in texts:
        for word in text.split():
            word_freqs["\u2581" + word] += 1
    return word_freqs

def word_to_symbols(word):
    return tuple(chr(b) for b in word.encode("utf-8"))


class BPETokenizer:
    def __init__(self):
        self.merges = []
        self.itos = []
        self.stoi = {}
        self.merge_rank = {}
        self.vocab_size = 0

    def train(self, texts, vocab_size, verbose=True):
        word_freqs = get_word_freqs(texts)
        splits = {word: list(word_to_symbols(word)) for word in word_freqs}

        vocab = set()
        for symbols in splits.values():
            vocab.update(symbols)
        vocab = sorted(vocab)

        self.itos = list(SPECIALS) + vocab
        self.stoi = {s: i for i, s in enumerate(self.itos)}

        pair_freq = Counter()
        pair_to_words = defaultdict(set)
        for word, symbols in splits.items():
            freq = word_freqs[word]
            for i in range(len(symbols) - 1):
                p = (symbols[i], symbols[i + 1])
                pair_freq[p] += freq
                pair_to_words[p].add(word)

        merges = []
        target_merges = vocab_size - len(self.itos)
        if target_merges < 0:
            raise ValueError(f"vocab_size {vocab_size} is smaller than the initial byte vocabulary ({len(self.itos)})")

        for step in tqdm(range(target_merges), desc="BPE merges"):
            if not pair_freq:
                if verbose:
                    print(f"No more pairs to merge after {step} merges; stopping early.")
                break

            best_pair, best_freq = None, -1
            for pair, freq in pair_freq.items():
                if freq > best_freq or (freq == best_freq and (best_pair is None or pair < best_pair)):
                    best_freq, best_pair = freq, pair

            new_symbol = "".join(best_pair)
            self.itos.append(new_symbol)
            self.stoi[new_symbol] = len(self.itos) - 1
            merges.append(list(best_pair))

            for word in list(pair_to_words[best_pair]):
                old_symbols = splits[word]
                freq = word_freqs[word]

                for i in range(len(old_symbols) - 1):
                    p = (old_symbols[i], old_symbols[i + 1])
                    pair_freq[p] -= freq
                    if pair_freq[p] <= 0:
                        del pair_freq[p]
                    pair_to_words[p].discard(word)
                    if not pair_to_words[p]:
                        del pair_to_words[p]

                new_symbols, i = [], 0
                while i < len(old_symbols):
                    if i < len(old_symbols) - 1 and (old_symbols[i], old_symbols[i + 1]) == best_pair:
                        new_symbols.append(new_symbol); i += 2
                    else:
                        new_symbols.append(old_symbols[i]); i += 1
                splits[word] = new_symbols

                for i in range(len(new_symbols) - 1):
                    p = (new_symbols[i], new_symbols[i + 1])
                    pair_freq[p] += freq
                    pair_to_words[p].add(word)

        self.merges = merges
        self.merge_rank = {tuple(m): i for i, m in enumerate(self.merges)}
        self.vocab_size = len(self.itos)
        if verbose:
            print(f"BPE training complete: {len(merges)} merges, final vocab size {self.vocab_size}")
        return self

    def tokenize(self, text):
        tokens = []
        for word in text.split():
            symbols = list(word_to_symbols("\u2581" + word))
            while len(symbols) > 1:
                pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
                ranked = [(self.merge_rank[p], i) for i, p in enumerate(pairs) if p in self.merge_rank]
                if not ranked:
                    break
                _, best_i = min(ranked)
                symbols = symbols[:best_i] + ["".join(symbols[best_i:best_i + 2])] + symbols[best_i + 2:]
            tokens.extend(symbols)
        return tokens


def detokenize(tokens):
    raw = "".join(tokens)
    byte_values = bytes(ord(c) for c in raw)
    text = byte_values.decode("utf-8", errors="replace")
    return text.replace("\u2581", " ").strip()

In [ ]:
%%writefile -a bpe_tok.py

# ---------------- Phase-1 helpers ----------------
import json as _json

def save_bpe(bpe, path):
    with open(path, "w", encoding="utf-8") as f:
        _json.dump({"merges": bpe.merges, "itos": bpe.itos, "vocab_size": bpe.vocab_size}, f, ensure_ascii=False)

def truncate_bpe_from(merges, itos, vocab_size):
    """BPE merges are sequential, so the first k merges of a big run ARE the size-(base+k) tokenizer."""
    n_base = len(itos) - len(merges)           # specials + initial byte symbols
    k = vocab_size - n_base
    assert 0 <= k <= len(merges), f"vocab_size must be between {n_base} and {len(itos)}"
    b = BPETokenizer()
    b.merges = [list(m) for m in merges[:k]]
    b.itos = list(itos[:n_base + k])
    b.stoi = {s: i for i, s in enumerate(b.itos)}
    b.merge_rank = {tuple(m): i for i, m in enumerate(b.merges)}
    b.vocab_size = len(b.itos)
    return b

def load_bpe(path, vocab_size=None):
    with open(path, encoding="utf-8") as f:
        d = _json.load(f)
    return truncate_bpe_from(d["merges"], d["itos"], vocab_size or len(d["itos"]))

def byte_coverage_text():
    """One small 'document' whose words contain every byte that can occur in UTF-8 text.
    Added once to the BPE training sample so that no character is ever <unk>: the tokenizer only
    gets base symbols for bytes it has seen. Each byte appears ~once, so it does not change the merges."""
    target = {b for b in range(0x80) if not chr(b).isspace()} | set(range(0x80, 0xC0)) | set(range(0xC2, 0xF5))
    chars, covered = [], set()
    for cp in range(1, 0x110000):
        if 0xD800 <= cp <= 0xDFFF or chr(cp).isspace():
            continue
        new = set(chr(cp).encode("utf-8")) & (target - covered)
        if new:
            chars.append(chr(cp)); covered |= new
            if covered == target:
                break
    return " ".join(chars)

class FastEncoder:
    """Caches word -> ids, so each distinct word goes through the merge loop only once."""
    def __init__(self, bpe, max_cache=2_000_000):
        self.bpe, self.stoi, self.cache, self.max_cache = bpe, bpe.stoi, {}, max_cache
        self.unk = bpe.stoi["<unk>"]
    def encode_word(self, w):
        ids = self.cache.get(w)
        if ids is None:
            ids = [self.stoi.get(t, self.unk) for t in self.bpe.tokenize(w)]
            if len(self.cache) < self.max_cache:
                self.cache[w] = ids
        return ids
    def encode(self, text):
        out = []
        for w in text.split():
            out.extend(self.encode_word(w))
        return out

## A5. Train BPE once at 32k on a 5M-word legal sample (≈10–20 min)
Watch the progress bar. If it shows fewer than ~30 merges/s, stop the cell, set `MAX_VOCAB = 24_000` and rerun.

In [ ]:
import importlib, bpe_tok
importlib.reload(bpe_tok)

BPE_SAMPLE_WORDS = 5_000_000
MAX_VOCAB = 32_000
total_train_words = sum(s["train_words"] for s in stats.values())
q = BPE_SAMPLE_WORDS / total_train_words          # same rate for every source -> sample follows the mix
rng = random.Random(SEED + 1)
sample = []
for name in FILES:
    with open(f"{TMP}/{name}.train.txt", encoding="utf-8") as f:
        sample += [ln for ln in f if rng.random() < q]
sample.append(bpe_tok.byte_coverage_text())       # guarantees every UTF-8 byte has a base token (no <unk>)
print(f"BPE sample: {len(sample):,} docs, {sum(len(s.split()) for s in sample):,} words")

t0 = time.time()
full_bpe = bpe_tok.BPETokenizer().train(sample, vocab_size=MAX_VOCAB)
print(f"Trained in {(time.time() - t0) / 60:.1f} min, vocab {full_bpe.vocab_size}")
bpe_tok.save_bpe(full_bpe, f"{WORK}/bpe_full.json")

## A6. Measurement table for choosing the vocabulary size
**How to read it:** a bigger vocabulary gives fewer tokens per word, so more text fits in each 1024-token context and each GPU-second, but the embedding and output layers grow by V × d. Choose the point where tokens/word stops falling much (the "knee"), and check that the `prompt+128` column stays well under 1024. Put this table and your reasoning in the report.

In [ ]:
val_docs = []
for name in FILES:
    with open(f"{TMP}/{name}.val.txt", encoding="utf-8") as f:
        val_docs += [ln for _, ln in zip(range(60), f)]
val_words = sum(len(d.split()) for d in val_docs)

print(f"{'vocab':>6} | {'tok/word':>8} | {'unk %':>6} | {'512-word prompt':>15} | {'prompt+128':>10} | "
      f"{'train tokens':>12} | {'embedding params d=512 / d=768':>30}")
for V in [8000, 12000, 16000, 24000, 32000]:
    if V > full_bpe.vocab_size:
        continue
    b = bpe_tok.truncate_bpe_from(full_bpe.merges, full_bpe.itos, V)
    enc = bpe_tok.FastEncoder(b)
    ids = [i for d in val_docs for i in enc.encode(d)]
    tpw = len(ids) / val_words
    unk = 100 * ids.count(b.stoi["<unk>"]) / len(ids)
    print(f"{V:>6} | {tpw:8.3f} | {unk:6.3f} | {512*tpw:15.0f} | {640*tpw:10.0f} | "
          f"{total_train_words*tpw/1e6:10.0f}M | {V*512/1e6:12.1f}M / {V*768/1e6:.1f}M")

## A7. YOUR choice of vocabulary size

In [ ]:
VOCAB_SIZE = None          # <-- YOUR choice from the A6 table, e.g. one of the rows
assert VOCAB_SIZE and VOCAB_SIZE <= 65535, "pick a vocab size (<= 65535 so ids fit in uint16)"
bpe = bpe_tok.truncate_bpe_from(full_bpe.merges, full_bpe.itos, VOCAB_SIZE)
bpe_tok.save_bpe(bpe, f"{WORK}/bpe.json")
print("Saved bpe.json with vocab", bpe.vocab_size)

## A8. Encode everything to `uint16` token files on all CPU cores (≈10–20 min)

In [ ]:
from multiprocessing import Pool

EOS_ID = bpe.stoi["</s>"]
_ENC = None

def _init_worker():
    global _ENC
    _ENC = bpe_tok.FastEncoder(bpe_tok.load_bpe(f"{WORK}/bpe.json"))

def _encode_lines(lines):
    out = []
    for ln in lines:
        out.extend(_ENC.encode(ln))
        out.append(EOS_ID)                 # document separator
    return np.asarray(out, dtype=np.uint16)

def _line_batches(paths, n=256):
    for p in paths:
        with open(p, encoding="utf-8") as f:
            batch = []
            for ln in f:
                batch.append(ln)
                if len(batch) == n:
                    yield batch; batch = []
            if batch:
                yield batch

def encode_files(paths, out_path):
    total, t0 = 0, time.time()
    with open(out_path, "wb") as fo, Pool(os.cpu_count(), initializer=_init_worker) as pool:
        for k, arr in enumerate(pool.imap(_encode_lines, _line_batches(paths), chunksize=2)):
            arr.tofile(fo); total += len(arr)
            if k % 200 == 0:
                print(f"  {os.path.basename(out_path)}: {total/1e6:,.0f}M tokens | {time.time()-t0:,.0f}s", flush=True)
    return total

n_val = encode_files([f"{TMP}/{n}.val.txt" for n in FILES], f"{WORK}/val.bin")
n_train = encode_files([f"{TMP}/{n}.train.txt" for n in FILES], f"{WORK}/train.bin")
print(f"train tokens: {n_train:,} | val tokens: {n_val:,}")

## A9. Local validation set shaped like the test (300 examples)

In [ ]:
VAL_EXAMPLES = {"supreme_court": 120, "high_courts": 100, "tribunals": 55, "acts": 25}
rng = random.Random(SEED + 2)
val_prompts, val_truth, per_source = {}, {}, {}
for name, n in VAL_EXAMPLES.items():
    with open(f"{TMP}/{name}.val.txt", encoding="utf-8") as f:
        docs = [ln.split() for ln in f]
    docs = [d for d in docs if len(d) >= 640]
    rng.shuffle(docs)
    for d in docs[:n]:
        s = rng.randint(0, len(d) - 640)          # random position inside the document, like the test
        k = str(len(val_prompts) + 1)
        val_prompts[k] = {"text": " ".join(d[s:s + 512])}
        val_truth[k] = {"text": " ".join(d[s + 512:s + 640])}
    per_source[name] = min(n, len(docs))
json.dump(val_prompts, open(f"{WORK}/val_prompts.json", "w", encoding="utf-8"), ensure_ascii=False)
json.dump(val_truth, open(f"{WORK}/val_truth.json", "w", encoding="utf-8"), ensure_ascii=False)
print("validation examples per source:", per_source, "| total:", len(val_prompts))

## A10. Sanity check + metadata

In [ ]:
arr = np.memmap(f"{WORK}/train.bin", dtype=np.uint16, mode="r")
print("Decoded start of train.bin:\n", bpe_tok.detokenize([bpe.itos[i] for i in arr[:150] if i >= 4]))
meta = {"stats": stats, "target_words": TARGET_WORDS, "vocab_size": VOCAB_SIZE,
        "train_tokens": int(n_train), "val_tokens": int(n_val),
        "filters": {"MIN_WORDS": MIN_WORDS, "MAX_DEVANAGARI": MAX_DEVANAGARI,
                    "MAX_PIPE_RATE": MAX_PIPE_RATE, "VAL_DOC_PROB": VAL_DOC_PROB}}
json.dump(meta, open(f"{WORK}/meta.json", "w"), indent=1)
!ls -lh /kaggle/working

## Save the outputs
**Save Version → Quick Save → Advanced Settings → "Always save output for this version" → Save.**
Then open the saved version and check that its **Output** tab lists `train.bin`, `bpe.json` and the other files.
If they are missing, set `VOCAB_SIZE` in A7 and use **Save & Run All** instead (reruns everything in the background, about 1 h).